# Build a Transformer from Scratch - Exercise Notebook

## Overview
This notebook will guide you through implementing a complete transformer language model step by step.

**What you'll build:**
- Multi-head attention mechanism
- Rotary positional embeddings (RoPE)
- Transformer decoder blocks
- Complete language model
- Training and evaluation pipeline

**Learning objectives:**
- Understand attention mechanism internals
- Master transformer architecture components
- Learn proper training practices

---

## Part 1: Setup and Imports

In [1]:
# TODO: Import required libraries
# - torch, torch.nn, torch.nn.functional
# - math, numpy
# - typing (Optional, Tuple)
# - torch.utils.data (Dataset, DataLoader)

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from typing import Optional, Tuple
from torch.utils.data import Dataset, DataLoader

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## Part 2: Dataset Preparation

Create a simple dataset that generates random token sequences for training.

In [2]:
# Constants
SEQ_LEN = 32
VOCAB_SIZE = 1000
BATCH_SIZE = 64
TRAIN_SAMPLES = 1000
VAL_SAMPLES = 200

class RandomSequenceDataset(Dataset):
    """Dataset that generates random token sequences."""
    
    def __init__(self, num_samples, seq_len, vocab_size):
        # TODO: Store parameters
        self.vocab_size = vocab_size
        self.num_samples = num_samples
        self.seq_len = seq_len

    def __len__(self):
        # TODO: Return number of samples
        return self.num_samples

    def __getitem__(self, idx):
        # TODO: Generate random input sequence x and target sequence y
        # Both should be of shape (seq_len,) with values in range [0, vocab_size)
        # In real LM, y would be x shifted by 1, but for simplicity we use random targets
        x = torch.randint(0, self.vocab_size, (self.seq_len))
        y = torch.randint(0, self.vocab_size, (self.seq_len))
        return x, y

# Create datasets and dataloaders
# TODO: Instantiate train and validation datasets
# TODO: Create DataLoaders with appropriate batch size and shuffle settings

train_dataset = RandomSequenceDataset(TRAIN_SAMPLES, SEQ_LEN, VOCAB_SIZE)
val_dataset = RandomSequenceDataset(VAL_SAMPLES, SEQ_LEN, VOCAB_SIZE)

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle=True)   # TODO
val_loader = DataLoader(val_dataset, batch_size = BATCH_SIZE, shuffle=True)   # TODO

## Part 3: Multi-Head Attention

Implement the core attention mechanism:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head attention mechanism.
    
    Args:
        d_model: Model dimension
        num_heads: Number of attention heads
        dropout: Dropout probability
    """
    
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1, max_len: int = 512):
        super().__init__()
        # TODO: Assert d_model is divisible by num_heads
        assert d_model % num_heads == 0
        
        # TODO: Store d_model, num_heads, and compute d_k = d_model // num_heads
        self.d_model = d_model
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        # TODO: Create linear projection layers for Q, K, V (d_model -> d_model)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.rope = RotaryPositionalEmbedding(d_model, max_len)
        # Hint: Use nn.Linear(d_model, d_model)
        
        # TODO: Create output projection layer (d_model -> d_model)
        
        # TODO: Create dropout layer
        self.dropout = nn.Dropout(dropout)
    
    def _shape_qkv(self, x: torch.tensor):
        batch_size, seq_len, _ = x.size()
        proj = x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        return proj.contiguous()

        
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            x: Input tensor of shape (batch_size, seq_len, d_model)
            mask: Optional attention mask
        
        Returns:
            Output tensor of shape (batch_size, seq_len, d_model)
        """
        batch_size, seq_len, _ = x.size()
        
        # TODO: Project input to Q, K, V
        # Q = self.W_q(x), etc.
        Q = self._shape_qkv(self.W_q(x))
        K = self._shape_qkv(self.W_k(x))
        V = self._shape_qkv(self.W_v(x))


        Q, K = self.rope(Q, K)
        # TODO: Reshape to (batch_size, num_heads, seq_len, d_k)
        # Hint: Use .view() and .transpose()
        # Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # TODO: Compute attention scores: Q @ K^T / sqrt(d_k)
        # Hint: Use torch.matmul() and K.transpose(-2, -1)
        scores = torch.matmul(Q, K.transpose(-2,-1)) / math.sqrt(self.d_k)
        
        # TODO: Apply mask if provided (set masked positions to -inf)
        # Hint: scores.masked_fill(mask == 0, float('-inf'))
        if mask:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # TODO: Apply softmax to get attention weights
        attn_weights = F.softmax(scores, dim=-1)
        # TODO: Apply dropout to attention weights
        attn_weights = self.dropout(attn_weights)
        # TODO: Multiply attention weights by V
        attn_output = torch.matmul(attn_weights, V)
        
        # TODO: Concatenate heads back together
        # Hint: .transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
        attn_output  = attn_output.transpose(1,2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # TODO: Apply output projection
        return self.W_o(attn_output)

## Part 4: Feed-Forward Network

Position-wise feed-forward network: FFN(x) = Linear(GELU(Linear(x)))

In [ ]:
class FeedForward(nn.Module):
    """
    Position-wise feed-forward network.
    
    Args:
        d_model: Model dimension
        d_ff: Hidden dimension (usually 4 * d_model)
        dropout: Dropout probability
    """
    
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        # TODO: Create first linear layer (d_model -> d_ff)
        self.layer1 = nn.Linear(d_model, d_ff)
        # TODO: Create second linear layer (d_ff -> d_model)
        self.layer2 = nn.Linear(d_ff, d_model)
        # TODO: Create dropout layer
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Apply first linear layer
        x = self.layer1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        return self.layer2(x)
        
        # TODO: Apply GELU activation
        # Hint: Use F.gelu()
        
        # TODO: Apply dropout
        
        # TODO: Apply second linear layer and return
        # pass

## Part 5: Layer Normalization

$$\text{LayerNorm}(x) = \gamma \frac{x - \mu}{\sigma + \epsilon} + \beta$$

In [ ]:
class LayerNorm(nn.Module):
    """
    Layer normalization.
    
    Args:
        d_model: Model dimension
        eps: Small constant for numerical stability
    """
    
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        # TODO: Create learnable scale parameter gamma (initialized to ones)
        # Hint: nn.Parameter(torch.ones(d_model))
        # TODO: Create learnable shift parameter beta (initialized to zeros)
        # TODO: Store eps
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Compute mean along last dimension
        # Hint: x.mean(-1, keepdim=True)
        mean = x.mean(-1, keepdim = True)
        # TODO: Compute standard deviation along last dimension
        std = x.std(-1, keepdim = True)
        # TODO: Normalize: (x - mean) / (std + eps)
        return self.gamma * (x-mean) / (std + eps) + self.beta
        # TODO: Apply affine transformation: gamma * normalized + beta
        # pass

## Part 6: Rotary Positional Embeddings (RoPE)

RoPE encodes position information by rotating query and key vectors.

In [ ]:
class RotaryPositionalEmbedding(nn.Module):
    """
    Rotary Positional Embeddings (RoPE).
    
    Args:
        d_model: Model dimension (must be even)
        max_len: Maximum sequence length
    """
    
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        assert d_model % 2 == 0
        inv_freq = 1.0 / (10000 ** (torch.arange(0, d_model, 2, dtype=torch.float)/d_model))
        self.register_buffer('inv_freq', inv_freq)

        t = torch.arange(0, max_len, dtype=torch.float)
        freqs = torch.outer(t, inv_freq) # every combination of t and inv_freq
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cached_cos', emb.cos())
        self.register_buffer('cached_sin', emb.sin())
    
    def rotate_half(self, x: torch.Tensor) -> torch.Tensor:
        """Rotates half the hidden dims of the input."""
        # TODO: Split x into two halves along last dimension
        # TODO: Return concatenation of [-x2, x1]
        d_model = x.shape[-1]
        x1, x2 = x[..., :d_model//2], x[..., d_model//2:]
        return torch.cat([-x2, x1])
    
    def forward(self, q: torch.Tensor, k: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Apply rotary embeddings to queries and keys.
        
        Args:
            q: Queries of shape (batch, num_heads, seq_len, d_k)
            k: Keys of shape (batch, num_heads, seq_len, d_k)
        
        Returns:
            Rotated queries and keys
        """
        seq_len = q.size(-2)
        cos = cached_cos[:seq_len, :].unsqueeze(0).unsqueeze(0)
        sin = cached_sin[:seq_len, :].unsqueeze(0).unsqueeze(0)
        q_rot = cos * q + self.rotate_half(q) * sin
        k_rot = cos * k + self.rotate_half(k) * sin
        return q_rot, k_rot

        

## Part 7: Causal Mask

Create a mask to prevent attention to future positions.

In [ ]:
def create_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """
    Create lower triangular causal mask.
    
    Args:
        seq_len: Sequence length
        device: Device to create mask on
    
    Returns:
        Mask of shape (1, 1, seq_len, seq_len)
    """
    # TODO: Create lower triangular matrix using torch.tril
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    # TODO: Add batch and head dimensions using .unsqueeze()
    return mask.unsqueeze(0).unsqueeze(0)

## Part 8: Transformer Decoder Block

Combine attention, feed-forward, and normalization with residual connections.

In [ ]:
class TransformerDecoderBlock(nn.Module):
    """
    Single transformer decoder block with pre-norm architecture.
    
    Args:
        d_model: Model dimension
        num_heads: Number of attention heads
        d_ff: Feed-forward hidden dimension
        dropout: Dropout probability
    """
    
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1, max_len):
        super().__init__()
        
        # TODO: Initialize multi-head attention
        self.attention = MultiHeadAttention(d_model, num_heads, dropout, max_len)
        # TODO: Initialize feed-forward network
        self.feed_forward = Feedforward(d_model, d_ff, dropout)
        # TODO: Initialize two layer norm layers
        self.layernorm1 = LayerNorm(d_model)
        self.layernorm2 = LayerNorm(d_model)
        # TODO: Initialize two dropout layers
        self.dropout1 = nn.Dropout(d_model)
        self.dropout2 = nn.Dropout(d_model)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass with pre-norm architecture:
        x = x + attention(norm(x))
        x = x + feedforward(norm(x))
        """
        # TODO: Apply attention with residual connection
        # 1. Normalize input
        # 2. Apply attention
        # 3. Apply dropout
        # 4. Add residual: x = x + ...
        
        # TODO: Apply feed-forward with residual connection
        # Same pattern as above
        h = self.attention(self.layernorm1(x), mask)
        x = x + self.dropout1(h) 

        h = self.feed_forward(self.layernorm2(x))
        x = x + self.dropout2(h)
        return x
        

## Part 9: Complete Transformer Language Model

In [ ]:
class TransformerLM(nn.Module):
    """
    Complete transformer language model.
    
    Args:
        vocab_size: Size of vocabulary
        d_model: Model dimension
        num_heads: Number of attention heads
        num_layers: Number of transformer blocks
        d_ff: Feed-forward hidden dimension
        max_len: Maximum sequence length
        dropout: Dropout probability
    """
    
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, d_ff: int, max_len: int = 512,
                 dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(d_model, num_heads, d_ff, dropout, max_len)
            for _ in range(num_layers)
        ])    
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNorm(d_model)
        self.output = nn.Linear(d_model, vocab_size)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            x: Input token IDs of shape (batch_size, seq_len)
            mask: Optional attention mask
        
        Returns:
            Logits of shape (batch_size, seq_len, vocab_size)
        """
        x = self.token_embedding(x) * math.sqrt(d_model)
        x = self.dropout(x)
        
        for layer in self.layers:
            x = layer(x, mask)

        x = self.norm(x)
        logits = self.output(x)
        return logits
       
    
    @torch.no_grad()
    def generate(self, prompt: torch.Tensor, max_new_tokens: int,
                 temperature: float = 1.0, top_k: Optional[int] = None) -> torch.Tensor:
        """
        Generate tokens autoregressively.
        
        Args:
            prompt: Initial sequence of shape (batch, seq_len)
            max_new_tokens: Number of tokens to generate
            temperature: Sampling temperature
            top_k: If set, only sample from top k tokens
        
        Returns:
            Extended sequence
        """
        self.eval()

        for _ in range(max_new_tokens):
            mask = create_causal_mask(seq_len, x.device)
            logits = self(x, mask)

            if top_k:
                v, _ = torch.top_k(x, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            prompt = torch.cat([prompt, next_token], dim=1)

        return prompt


## Part 10: Training Setup

In [3]:
model = TransformerLM(...).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(optimizer, T_MAX=100, eta=1e-6)


NameError: name 'TransformerLM' is not defined

## Part 11: Training Loop

In [ ]:
epochs = 50
for epoch in epochs:
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_dataloader:
        batch_x.to(device), batch_y.to(device)

        mask = create_causal_mask(batch_x.shape[1], device)
        logits = model(batch_x, mask)

        loss = criterion(logits.view(-1, batch_x.shape[1]), batch_y.view(-1))

        # backward pass
        optimizer.zero_grad()
        loss.backwards()

        optimizer.step()
        total_loss += loss

    scheduler.step()
    avg_loss = total_loss / len(train_loader)
    perplexity = math.exp(avg_loss)
    

        

## Part 12: Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, data_loader, criterion):
    """
    Evaluate model on validation data.
    
    Returns:
        avg_loss: Average loss
        perplexity: exp(avg_loss)
    """
    # TODO: Set model to eval mode
    model.eval()
    total_loss = 0

    for batch_x, batch_y in data_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        mask = create_causal_mask(batch_x.shape[1])
        logits = model(batch_x, mask)

        loss = criterion(logis.view(-1, batch_x.shape[1]), batch_y.view(-1))
        # backward pass
        optimizer.zero_grad()
        loss.backward()

        # Gradient clipping
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # update weights
        # optimizer.step()
        total_loss += loss
    
    # TODO: Loop through validation data
    avg_loss = total_loss / len(data_loader)
    perplexity = math.exp(avg_loss)
    # TODO: Compute loss (no gradient tracking)
    return avg_loss, perplexity
    # TODO: Return average loss and perplexity

# TODO: Run evaluation
# val_loss, val_perplexity = evaluate(model, val_loader, criterion)
val_loss, val_perplexity = evaluate(model, val_loader, criterion)

## Part 13: Text Generation

In [ ]:
# Test generation
# TODO: Create a random prompt
prompt = torch.randint(0, VOCAB_SIZE, (1, 10)).to(device)

# TODO: Generate with different settings
# 1. Low temperature (more deterministic)
generated = model.generate(prompt, max_new_tokens=20, temperature=0.1)
# 2. High temperature (more random)
# 3. Top-k sampling

---

## Checklist

Make sure you've implemented:

- [ ] Dataset class with random sequences
- [ ] Multi-head attention mechanism
- [ ] Feed-forward network
- [ ] Layer normalization
- [ ] Rotary positional embeddings
- [ ] Causal masking function
- [ ] Transformer decoder block
- [ ] Complete language model
- [ ] Generate method
- [ ] Training loop with gradient clipping
- [ ] Evaluation function
- [ ] Text generation with sampling

## Common Issues

1. **Shape mismatches**: Print tensor shapes at each step
2. **NaN loss**: Check gradient clipping, learning rate, mask application
3. **Slow training**: Reduce batch size, use smaller model
4. **Out of memory**: Reduce batch_size, seq_len, or model size

## Next Steps

Once working:
- Try KV caching for faster generation
- Implement grouped-query attention (GQA)
- Add Flash Attention
- Train on real text data
- Experiment with model size